# Imports necessários

In [1]:
import pandas as pd 
import numpy as np
import json
import os
import time

from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, ValidationError
from typing import Literal


# Obtenção dos dados

In [2]:
with open("../dados/dados_nivel_2.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

print(dados.keys())

dict_keys(['taxa_cambio_usd_brl', 'operacoes'])


In [3]:
dados

{'taxa_cambio_usd_brl': 5.4,
 'operacoes': [{'id': 'OP-00133',
   'cliente_id': 'CLI-014',
   'data': '2026-03-06',
   'valor': 23640.97,
   'moeda': 'BRL',
   'canal': 'pix',
   'tipo': 'pagamento',
   'contraparte': 'Mirante Transportes ME',
   'observacao': ''},
  {'id': 'OP-00103',
   'cliente_id': 'CLI-011',
   'data': '2026-03-04',
   'valor': 9447.52,
   'moeda': 'BRL',
   'canal': 'cartao',
   'tipo': 'saque',
   'contraparte': 'Quartzo Industria SA',
   'observacao': ''},
  {'id': 'OP-00223',
   'cliente_id': 'CLI-023',
   'data': '2026-04-13',
   'valor': 2891.48,
   'moeda': 'BRL',
   'canal': 'ted',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Gama Importacao SA',
   'observacao': ''},
  {'id': 'OP-00265',
   'cliente_id': 'CLI-028',
   'data': '2026-04-23',
   'valor': 5636.46,
   'moeda': 'BRL',
   'canal': 'pix',
   'tipo': 'saque',
   'contraparte': 'Nauta Atacado ME',
   'observacao': ''},
  {'id': 'OP-00099',
   'cliente_id': 'CLI-010',
   'data': '2026-03-2

In [4]:
# "taxa_cambio_usd_brl" é um campo único
list(dados.keys()).count("taxa_cambio_usd_brl")

1

In [5]:
TAXA_CAMBIO = float(dados["taxa_cambio_usd_brl"])
operacoes = dados["operacoes"]

* Em 'operacoes', todos os registros possuem a mesma estrutura?

In [6]:
def validar_estrutura_operacoes(operacoes):
    if not operacoes:
        raise ValueError("A lista de operações está vazia.")

    chaves_esperadas = set(operacoes[0].keys())
    inconsistencias = []

    for i, operacao in enumerate(operacoes):
        chaves = set(operacao.keys())

        if chaves != chaves_esperadas:
            inconsistencias.append({
                "indice": i,
                "faltando": sorted(chaves_esperadas - chaves),
                "a_mais": sorted(chaves - chaves_esperadas)
            })

    return {
        "valido": len(inconsistencias) == 0,
        "chaves_esperadas": chaves_esperadas,
        "inconsistencias": inconsistencias
    }

In [7]:
resultado = validar_estrutura_operacoes(operacoes)

resultado

{'valido': True,
 'chaves_esperadas': {'canal',
  'cliente_id',
  'contraparte',
  'data',
  'id',
  'moeda',
  'observacao',
  'tipo',
  'valor'},
 'inconsistencias': []}

#### Todos os registros possuem as mesmas chaves, portanto, podemos criar o DataFrame somente com 'operacoes'.

In [8]:
df = pd.DataFrame(operacoes)

# como é um dataframe pequeno, prefiro visualizá-lo todo.
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-00133,CLI-014,2026-03-06,23640.97,BRL,pix,pagamento,Mirante Transportes ME,
1,OP-00103,CLI-011,2026-03-04,9447.52,BRL,cartao,saque,Quartzo Industria SA,
2,OP-00223,CLI-023,2026-04-13,2891.48,BRL,ted,transferencia_enviada,Gama Importacao SA,
3,OP-00265,CLI-028,2026-04-23,5636.46,BRL,pix,saque,Nauta Atacado ME,
4,OP-00099,CLI-010,2026-03-20,6641.24,BRL,ted,pagamento,Delta Trading LTDA,
...,...,...,...,...,...,...,...,...,...
317,OP-00195,CLI-021,2026-03-14,1627.63,BRL,especie,transferencia_recebida,Duna Servicos ME,
318,OP-00076,CLI-008,2026-04-23,6332.42,BRL,cartao,transferencia_recebida,Aurora Assessoria LTDA,
319,OP-00112,CLI-012,2026-05-09,772.58,BRL,boleto,pagamento,Estrela Transportes LTDA,
320,OP-00281,CLI-029,2026-03-11,3965.12,BRL,especie,pagamento,Estrela Consultoria SA,


#### Pontos de qualidade dos dados:
* dados nos formatos adequados? existem Nans? existem duplicatas?
* ID's são únicos?
* datas estão todas no mesmo formato? são todas datas válidas?
* existem dados diferentes que representam a mesma coisa? por exemplo: pix e PIX, deposito e depósito...


# Entendimento da Base

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           322 non-null    object 
 1   cliente_id   322 non-null    object 
 2   data         315 non-null    object 
 3   valor        322 non-null    float64
 4   moeda        322 non-null    object 
 5   canal        322 non-null    object 
 6   tipo         322 non-null    object 
 7   contraparte  322 non-null    object 
 8   observacao   322 non-null    object 
dtypes: float64(1), object(8)
memory usage: 22.8+ KB


In [11]:
df[df.duplicated(keep=False)].sort_values(by='id')

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
63,OP-00040,CLI-005,None,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
186,OP-00040,CLI-005,None,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
10,OP-00160,CLI-017,2026-03-19,3712.72,BRL,especie,saque,Lumen Industria ME,
143,OP-00160,CLI-017,2026-03-19,3712.72,BRL,especie,saque,Lumen Industria ME,
54,OP-00214,CLI-023,2026-03-20,1335.29,BRL,ted,saque,Mirante Consultoria SA,
253,OP-00214,CLI-023,2026-03-20,1335.29,BRL,ted,saque,Mirante Consultoria SA,
7,OP-00269,CLI-028,2026-05-23,6913.84,BRL,cartao,transferencia_enviada,Farol Distribuidora LTDA,
118,OP-00269,CLI-028,2026-05-23,6913.84,BRL,cartao,transferencia_enviada,Farol Distribuidora LTDA,
248,OP-00272,CLI-028,2026-03-27,6076.89,BRL,boleto,transferencia_enviada,Lumen Servicos SA,
266,OP-00272,CLI-028,2026-03-27,6076.89,BRL,boleto,transferencia_enviada,Lumen Servicos SA,


In [12]:
df[df['data'].isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
63,OP-00040,CLI-005,None,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
180,OP-00194,CLI-021,None,3440.82,BRL,ted,transferencia_recebida,Farol Varejo ME,data nao capturada pelo sistema
186,OP-00040,CLI-005,None,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
196,OP-00163,CLI-017,None,2162.68,BRL,pix,transferencia_recebida,Alfa Assessoria SA,data nao capturada pelo sistema
262,OP-00277,CLI-029,None,12407.61,BRL,boleto,deposito,Pampa Varejo LTDA,data nao capturada pelo sistema
271,OP-00048,CLI-005,None,1305.54,BRL,ted,deposito,Cristal Comercio SA,data nao capturada pelo sistema
273,OP-00006,CLI-001,None,9490.55,BRL,ted,saque,Estrela Transportes LTDA,data nao capturada pelo sistema


In [13]:
df = df.drop_duplicates(keep="first")

In [14]:
df['valor'] = df['valor'].astype(float)

C:\Users\yghor\AppData\Local\Temp\ipykernel_17432\1997309100.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['valor'] = df['valor'].astype(float)


In [15]:
df.shape

(317, 9)

## id

In [ ]:
df['id'].nunique()

317

## data

In [ ]:
df[~df["data"].str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
63,OP-00040,CLI-005,None,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
180,OP-00194,CLI-021,None,3440.82,BRL,ted,transferencia_recebida,Farol Varejo ME,data nao capturada pelo sistema
196,OP-00163,CLI-017,None,2162.68,BRL,pix,transferencia_recebida,Alfa Assessoria SA,data nao capturada pelo sistema
262,OP-00277,CLI-029,None,12407.61,BRL,boleto,deposito,Pampa Varejo LTDA,data nao capturada pelo sistema
271,OP-00048,CLI-005,None,1305.54,BRL,ted,deposito,Cristal Comercio SA,data nao capturada pelo sistema
273,OP-00006,CLI-001,None,9490.55,BRL,ted,saque,Estrela Transportes LTDA,data nao capturada pelo sistema


In [19]:
# todas são datas válidas? todas as datas, com exceção do caso encontrado, são data válidas!
datas = pd.to_datetime(df["data"], format="%Y-%m-%d", errors="coerce")

df[datas.isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
63,OP-00040,CLI-005,None,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
180,OP-00194,CLI-021,None,3440.82,BRL,ted,transferencia_recebida,Farol Varejo ME,data nao capturada pelo sistema
196,OP-00163,CLI-017,None,2162.68,BRL,pix,transferencia_recebida,Alfa Assessoria SA,data nao capturada pelo sistema
262,OP-00277,CLI-029,None,12407.61,BRL,boleto,deposito,Pampa Varejo LTDA,data nao capturada pelo sistema
271,OP-00048,CLI-005,None,1305.54,BRL,ted,deposito,Cristal Comercio SA,data nao capturada pelo sistema
273,OP-00006,CLI-001,None,9490.55,BRL,ted,saque,Estrela Transportes LTDA,data nao capturada pelo sistema


In [20]:
df["data"] = pd.to_datetime(df["data"])

C:\Users\yghor\AppData\Local\Temp\ipykernel_17432\1343936737.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["data"] = pd.to_datetime(df["data"])


## moeda

In [ ]:
# somente sete registros em USD
df['moeda'].value_counts()

moeda
BRL    310
USD      7
Name: count, dtype: int64

## valor_brl

In [23]:
# criamos uma coluna para normalizar valores para BRL
df["valor_brl"] = df["valor"]

df.loc[df["moeda"] == "USD", "valor_brl"] = (
    df.loc[df["moeda"] == "USD", "valor"] * TAXA_CAMBIO
)

C:\Users\yghor\AppData\Local\Temp\ipykernel_17432\184986511.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["valor_brl"] = df["valor"]


In [26]:
df[['moeda', 'valor', 'valor_brl']].sample(10)

,moeda,valor,valor_brl
211,BRL,5776.47,5776.470
19,BRL,13660.65,13660.650
127,USD,12638.36,68247.144
298,BRL,10938.75,10938.750
8,BRL,3194.61,3194.610
294,BRL,783.35,783.350
154,BRL,6434.36,6434.360
86,BRL,1999.29,1999.290
289,BRL,2464.02,2464.020
278,BRL,5279.94,5279.940


## canal

In [27]:
# não há valores anormais
df['canal'].value_counts()

canal
ted        81
especie    65
pix        61
cartao     57
boleto     53
Name: count, dtype: int64

## tipo

In [28]:
# não há valores anormais
df['tipo'].value_counts()

tipo
deposito                  69
pagamento                 68
transferencia_enviada     65
transferencia_recebida    60
saque                     55
Name: count, dtype: int64

# Investigações

## Volume por cliente

In [32]:
volume_cliente = (
    df.groupby("cliente_id")["valor_brl"]
      .sum()
      .reset_index(name="volume_total_brl")
)

volume_cliente

,cliente_id,volume_total_brl
0,CLI-001,47947.810
1,CLI-002,107965.380
2,CLI-003,102241.090
3,CLI-004,25968.570
4,CLI-005,64742.660
5,CLI-006,40996.150
6,CLI-007,58601.430
7,CLI-008,42394.050
8,CLI-009,47535.950
9,CLI-010,46607.100


## Operações por Canal

In [33]:
# essa informação deveria ser feita por cliente?

operacoes_canal = (
    df.groupby("canal")
      .size()
      .reset_index(name="quantidade_operacoes")
)

operacoes_canal

,canal,quantidade_operacoes
0,boleto,53
1,cartao,57
2,especie,65
3,pix,61
4,ted,81


## Regra 1 — Fracionamento

In [56]:
# Precisamos saber quantidade de operações, soma dos valores e maior operação para cada par cliente-data
fracionamento = (
    df.groupby(["cliente_id", "data"])
      .agg(
          quantidade_operacoes=("id", "count"),
          soma_valor_brl=("valor_brl", "sum"),
          maior_operacao_brl=("valor_brl", "max")
      )
      .reset_index()
)

# Aplicando a regra
fracionamento["fracionamento"] = (
    (fracionamento["quantidade_operacoes"] >= 3) &
    (fracionamento["soma_valor_brl"] > 50_000) &
    (fracionamento["maior_operacao_brl"] < 20_000)
)

In [ ]:
clientes_sem_data = df[df['data'].isna()]['cliente_id'].unique()
fracionamento[(fracionamento['cliente_id'].isin(clientes_sem_data))]

,cliente_id,data,quantidade_operacoes,soma_valor_brl,maior_operacao_brl,fracionamento
0,CLI-001,2026-03-06,1,668.120,668.120,False
1,CLI-001,2026-03-12,1,1605.710,1605.710,False
2,CLI-001,2026-03-15,1,25110.150,25110.150,False
3,CLI-001,2026-05-13,1,535.910,535.910,False
4,CLI-001,2026-05-14,1,2060.470,2060.470,False
5,CLI-001,2026-05-16,1,1608.240,1608.240,False
6,CLI-001,2026-05-19,1,241.970,241.970,False
7,CLI-001,2026-05-25,1,1610.070,1610.070,False
8,CLI-001,2026-05-26,1,5016.620,5016.620,False
39,CLI-005,2026-03-14,2,32631.360,30743.970,False


In [61]:
df[df['data'].isna()].groupby('cliente_id').agg({'valor': 'count'})

,valor
cliente_id,
CLI-001,1
CLI-005,2
CLI-017,1
CLI-021,1
CLI-029,1


CLI-005 é o único cliente com data Nan que poderia entrar no criterio de fracionamento se a data nan fosse a mesma dessas demais transações. Os demais não possuem operações suficientes, ou já estão com flag.

In [63]:
fracionamento[(fracionamento['cliente_id'] == 'CLI-005') & (fracionamento['quantidade_operacoes'] == 2)]

,cliente_id,data,quantidade_operacoes,soma_valor_brl,maior_operacao_brl,fracionamento
39,CLI-005,2026-03-14,2,32631.36,30743.97,False


Mesmo considerando as duas transações sem data, o valor total não supera 50000 nem o maior valor é limitado em 20000.

In [64]:
df[(df['data'].isna()) & (df['cliente_id'] == 'CLI-005')]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,contraparte_normalizada
63,OP-00040,CLI-005,NaT,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema,3025.05,solar importacao ltda
271,OP-00048,CLI-005,NaT,1305.54,BRL,ted,deposito,Cristal Comercio SA,data nao capturada pelo sistema,1305.54,cristal comercio sa


In [65]:
clientes_fracionamento = (
    fracionamento.loc[
        fracionamento["fracionamento"],
        "cliente_id"
    ]
    .unique()
)

df["flag_fracionamento"] = (
    df["cliente_id"].isin(clientes_fracionamento)
)

C:\Users\yghor\AppData\Local\Temp\ipykernel_17432\787473797.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["flag_fracionamento"] = (


## Regra 2 - Valor atípico

In [66]:
estatisticas_cliente = (
    df.groupby("cliente_id")["valor_brl"]
      .agg(
          quantidade_operacoes="count",
          mediana_brl="median"
      )
      .reset_index()
)

In [68]:
estatisticas_cliente["5x_mediana_brl"] = 5 * estatisticas_cliente["mediana_brl"]

In [69]:
df = df.merge(
    estatisticas_cliente,
    on="cliente_id",
    how="left"
)

In [70]:
df[['cliente_id', 'quantidade_operacoes', 'valor_brl', 'mediana_brl', '5x_mediana_brl']].sample(5)

,cliente_id,quantidade_operacoes,valor_brl,mediana_brl,5x_mediana_brl
234,CLI-020,9,1398.45,3397.720,16988.600
10,CLI-017,13,3712.72,9593.320,47966.600
315,CLI-029,16,3965.12,10337.485,51687.425
146,CLI-013,10,3796.46,3146.225,15731.125
124,CLI-023,12,1383.42,3241.160,16205.800


In [71]:
df["flag_valor_atipico"] = (
    (df["quantidade_operacoes"] >= 4) &
    (df["valor_brl"] > df["5x_mediana_brl"])
)

In [72]:
df.loc[
    df["flag_valor_atipico"],
    [
        "id",
        "cliente_id",
        "quantidade_operacoes",
        "mediana_brl",
        "5x_mediana_brl",
        "valor_brl",
        "flag_valor_atipico"
    ]
]

,id,cliente_id,quantidade_operacoes,mediana_brl,5x_mediana_brl,valor_brl,flag_valor_atipico
0,OP-00133,CLI-014,11,2308.410,11542.050,23640.970,True
18,OP-00197,CLI-021,10,2832.545,14162.725,15785.390,True
19,OP-00129,CLI-014,11,2308.410,11542.050,13660.650,True
24,OP-00253,CLI-026,12,2032.930,10164.650,21261.010,True
52,OP-00219,CLI-023,12,3241.160,16205.800,41768.170,True
82,OP-00008,CLI-001,10,1609.155,8045.775,25110.150,True
110,OP-00049,CLI-005,11,2144.180,10720.900,11988.170,True
117,OP-00310,CLI-013,10,3146.225,15731.125,28487.760,True
121,OP-00312,CLI-022,11,1909.890,9549.450,30894.858,True
126,OP-00316,CLI-024,11,2740.780,13703.900,68247.144,True


# Separando top 10 clientes mais marcados

In [74]:
# Sinalização de fracionamento: 1 por cliente (é constante nas linhas dele)
fracionamento_por_cliente = df.groupby('cliente_id')['flag_fracionamento'].max().astype(int)

# Sinalização de valor atípico: conta quantas operações foram flagadas
atipico_por_cliente = df.groupby('cliente_id')['flag_valor_atipico'].sum().astype(int)

# Volume total por cliente (usando valor_brl, já normalizado)
volume_por_cliente = df.groupby('cliente_id')['valor_brl'].sum()

# Combina tudo
resumo_sinalizacoes = pd.DataFrame({
    'sinalizacoes': fracionamento_por_cliente + atipico_por_cliente,
    'volume_total_brl': volume_por_cliente
})

top_10 = resumo_sinalizacoes.sort_values(
    by=['sinalizacoes', 'volume_total_brl'],
    ascending=[False, False]
).head(10)

top_10

,sinalizacoes,volume_total_brl
cliente_id,,
CLI-014,3,80629.990
CLI-023,2,148535.016
CLI-028,2,88750.800
CLI-013,2,81730.990
CLI-005,2,64742.660
CLI-026,2,54729.280
CLI-001,2,47947.810
CLI-029,1,191385.766
CLI-017,1,121391.370


## Configurações 

In [41]:
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

## Funções

In [42]:
def cria_resumo_cliente(df, cliente_id):
    df_cliente = df.loc[df["cliente_id"] == cliente_id].copy()

    if df_cliente.empty:
        raise ValueError(f"Cliente {cliente_id} não encontrado.")

    # Garantir ordenação temporal
    df_cliente["data"] = pd.to_datetime(df_cliente["data"])
    df_cliente = df_cliente.sort_values("data") 

    intervalos = (
            df_cliente["data"]
            .diff()
            .dt.days
            .dropna()
        )

    operacoes_por_data = (
            df_cliente
            .groupby("data")
            .size()
        )


    resumo_cliente = {
        # Indicadores gerais
        "quantidade_operacoes": len(df_cliente),
        "volume_total_brl": df_cliente["valor_brl"].sum(),
        "mediana_valor_brl": df_cliente["valor_brl"].median(),

        # Resultados das regras
        "flag_fracionamento": df_cliente["flag_fracionamento"].any(),
        "quantidade_valor_atipico": (
            df_cliente["flag_valor_atipico"].sum()
        ),

        # Diversidade operacional
        "n_canais": df_cliente["canal"].nunique(),
        "n_tipos_operacao": df_cliente["tipo"].nunique(),
        "n_contrapartes": df_cliente["contraparte"].nunique(),

        # Comportamento temporal
        "menor_intervalo_dias": (
            intervalos.min()
            if not intervalos.empty
            else None
        ),

        "maior_quantidade_operacoes_mesma_data": (
            operacoes_por_data.max()
        ),
    }

    return resumo_cliente

In [43]:
def cria_prompt_1(resumo_cliente):
    return f"""
                Analise o comportamento do cliente abaixo com base exclusivamente nos indicadores fornecidos, 
                e produza um parecer sobre o comportamento do cliente em relação à lavagem de dinheiro.

                IMPORTANTE:
                - Os indicadores foram calculados previamente por um sistema determinístico.
                - NÃO faça novos cálculos.
                - NÃO altere os valores fornecidos.
                - NÃO invente informações que não estejam no contexto.
                - NÃO compare valores.
                - NÃO sugira cálculos adicionais, dados faltantes ou próximos passos fora do que foi pedido.
                - Sua função é interpretar os indicadores e redigir um parecer.

                Quantidade de operações: {resumo_cliente["quantidade_operacoes"]}
                Volume total (BRL): {resumo_cliente["volume_total_brl"]}
                Possui flag de fracionamento: {resumo_cliente["flag_fracionamento"]}
                Quantidade de valores atípicos: {resumo_cliente["quantidade_valor_atipico"]}

                Retorne SOMENTE um JSON válido, sem markdown ou texto adicional,
                com exatamente os seguintes campos:

                {{
                    "nivel_risco": "baixo|médio|alto",
                    "tipologia_suspeita": "possível tipologia ou ausência de tipologia evidente",
                    "red_flags": ["sinal 1", "sinal 2", ...],
                    "justificativa": "justificativa objetiva da classificação"
                }}
            """

In [44]:
def cria_prompt_2(resumo_cliente):
    return f"""
                Você é um analista de Prevenção à Lavagem de Dinheiro (PLD) de um banco.

                Analise o comportamento do cliente abaixo com base exclusivamente
                nos indicadores fornecidos e produza um parecer sobre o comportamento
                do cliente.

                IMPORTANTE:
                - Todos os indicadores foram calculados previamente por um sistema
                determinístico.
                - NÃO faça novos cálculos.
                - NÃO altere os valores fornecidos.
                - NÃO invente informações que não estejam no contexto.
                - NÃO compare valores.
                - NÃO tente reconstruir ou validar as regras determinísticas.
                - NÃO sugira cálculos adicionais, dados faltantes ou próximos passos.
                - Sua função é interpretar os padrões apresentados e redigir um parecer.

                INDICADORES GERAIS

                Quantidade de operações:
                {resumo_cliente["quantidade_operacoes"]}

                Volume total (BRL):
                {resumo_cliente["volume_total_brl"]}

                Mediana dos valores (BRL):
                {resumo_cliente["mediana_valor_brl"]}

                RESULTADOS DAS REGRAS

                Possui flag de fracionamento:
                {resumo_cliente["flag_fracionamento"]}

                Quantidade de valores atípicos:
                {resumo_cliente["quantidade_valor_atipico"]}

                DIVERSIDADE OPERACIONAL

                Número de canais distintos:
                {resumo_cliente["n_canais"]}

                Número de tipos de operação distintos:
                {resumo_cliente["n_tipos_operacao"]}

                Número de contrapartes distintas:
                {resumo_cliente["n_contrapartes"]}

                COMPORTAMENTO TEMPORAL

                Maior quantidade de operações realizadas na mesma data:
                {resumo_cliente["maior_quantidade_operacoes_mesma_data"]}

                Com base exclusivamente nessas informações, produza um parecer
                sobre o comportamento do cliente.

                Retorne SOMENTE um JSON válido, sem markdown ou texto adicional,
                com exatamente os seguintes campos:

                {{
                    "nivel_risco": "baixo|médio|alto",
                    "tipologia_suspeita": "possível tipologia ou ausência de tipologia evidente",
                    "red_flags": ["sinal 1", "sinal 2", ...],
                    "justificativa": "justificativa objetiva da classificação"
                }}
            """

In [45]:
def consultar_llm(prompt):
    inicio = time.perf_counter()

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    tempo_resposta = time.perf_counter() - inicio

    usage = response.usage_metadata

    return {
        "resposta": response.text,
        "tempo_segundos": tempo_resposta,
        "tokens_entrada": usage.prompt_token_count,
        "tokens_saida": usage.candidates_token_count,
        "tokens_raciocinio": usage.thoughts_token_count or 0,
        "tokens_total": usage.total_token_count
    }

In [46]:
class Parecer(BaseModel):
    nivel_risco: Literal["baixo", "médio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

def validar_parecer(resposta):
    try:
        dados = json.loads(resposta)
        parecer = Parecer.model_validate(dados)

        return parecer, None

    except json.JSONDecodeError as erro:    
        return None, f"JSON inválido: {erro}"

    except ValidationError as erro:
        return None, f"Schema inválido: {erro}"

In [47]:
def gerar_parecer(prompt, max_tentativas=2):
    resultados = []

    prompt_atual = prompt

    for tentativa in range(1, max_tentativas + 1):

        resultado_llm = consultar_llm(prompt_atual)

        parecer, erro = validar_parecer(
            resultado_llm["resposta"]
        )

        resultados.append({
            "tentativa": tentativa,
            **resultado_llm,
            "valido": parecer is not None,
            "erro": erro
        })

        if parecer is not None:
            return {
                "parecer": parecer,
                "tentativas": resultados
            }

        # Se falhou, prepara uma nova tentativa
        prompt_atual = f"""
                            A resposta anterior não respeitou o formato solicitado.

                            Erro encontrado:
                            {erro}

                            Corrija SOMENTE o formato da resposta.

                            Retorne SOMENTE um JSON válido, sem markdown e sem texto adicional,
                            contendo exatamente:

                            {{
                                "nivel_risco": "baixo|médio|alto",
                                "tipologia_suspeita": "string",
                                "red_flags": ["string"],
                                "justificativa": "string"
                            }}
                        """

    return {
        "parecer": None,
        "tentativas": resultados
    }

In [48]:
def analisar_cliente(df, cliente_id):
    # ============================================================
    # 1. Criar resumo determinístico com pandas
    # ============================================================

    resumo_cliente = cria_resumo_cliente(
        df=df,
        cliente_id=cliente_id
    )

    # ============================================================
    # 2. Criar os dois prompts
    # ============================================================

    prompt_1 = cria_prompt_1(
        resumo_cliente=resumo_cliente
    )

    prompt_2 = cria_prompt_2(
        resumo_cliente=resumo_cliente
    )

    # ============================================================
    # 3. Executar os dois prompts
    # ============================================================

    resultado_1 = gerar_parecer(prompt_1)

    resultado_2 = gerar_parecer(prompt_2)

    # ============================================================
    # 4. Comparar os resultados
    # ============================================================

    comparacao = pd.DataFrame([
        {
            "prompt": "Prompt 1",
            "tempo_s": sum(
                t["tempo_segundos"]
                for t in resultado_1["tentativas"]
            ),
            "tokens_entrada": sum(
                t["tokens_entrada"]
                for t in resultado_1["tentativas"]
            ),
            "tokens_saida": sum(
                t["tokens_saida"]
                for t in resultado_1["tentativas"]
            ),
            "tokens_total": sum(
                t["tokens_total"]
                for t in resultado_1["tentativas"]
            ),
            "nivel_risco": (
                resultado_1["parecer"].nivel_risco
                if resultado_1["parecer"]
                else None
            ),
        },
        {
            "prompt": "Prompt 2",
            "tempo_s": sum(
                t["tempo_segundos"]
                for t in resultado_2["tentativas"]
            ),
            "tokens_entrada": sum(
                t["tokens_entrada"]
                for t in resultado_2["tentativas"]
            ),
            "tokens_saida": sum(
                t["tokens_saida"]
                for t in resultado_2["tentativas"]
            ),
            "tokens_total": sum(
                t["tokens_total"]
                for t in resultado_2["tentativas"]
            ),
            "nivel_risco": (
                resultado_2["parecer"].nivel_risco
                if resultado_2["parecer"]
                else None
            ),
        }
    ])

    # ============================================================
    # 5. Registrar todas as tentativas
    # ============================================================

    tentativas = []

    for resultado, nome_prompt in [
        (resultado_1, "Prompt 1"),
        (resultado_2, "Prompt 2")
    ]:
        for tentativa in resultado["tentativas"]:
            tentativas.append({
                "cliente_id": cliente_id,
                "prompt": nome_prompt,
                **tentativa
            })

    df_tentativas = pd.DataFrame(tentativas)

    # ============================================================
    # 6. Retornar tudo
    # ============================================================

    return {
        "cliente_id": cliente_id,
        "resumo_cliente": resumo_cliente,
        "prompt_1": prompt_1,
        "prompt_2": prompt_2,
        "resultado_1": resultado_1,
        "resultado_2": resultado_2,
        "comparacao": comparacao,
        "df_tentativas": df_tentativas
    }

O cliente CLI-A-1 foi escolhido por ter flag_fracionamento.

In [49]:
cli_a_1 = analisar_cliente(df, 'CLI-A-1')

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Resultado da chamada usando o Prompt 1

In [50]:
cli_a_1["resultado_1"]["parecer"].model_dump()

{'nivel_risco': 'médio',
 'tipologia_suspeita': 'Fracionamento de operações (Smurfing)',
 'red_flags': ['Presença de flag de fracionamento'],
 'justificativa': 'O cliente realizou 4 operações com volume total de 57500.0 BRL e apresenta a flag de fracionamento ativada, o que aponta para um comportamento de potencial divisão de recursos, embora a quantidade de valores atípicos seja igual a 0.'}

Resultado da chamada usando o Prompt 2

In [51]:
cli_a_1["resultado_2"]["parecer"].model_dump()

{'nivel_risco': 'alto',
 'tipologia_suspeita': 'Fracionamento (Smurfing)',
 'red_flags': ['Presença de flag de fracionamento de operações',
  'Alta concentração de operações na mesma data',
  'Utilização de múltiplos canais e contrapartes distintas'],
 'justificativa': 'O cliente apresentou flag positiva para fracionamento de operações, realizando 3 de suas 4 movimentações em uma mesma data. A conduta envolveu um volume total de R$ 57.500,00 distribuído entre 3 canais e 3 contrapartes distintas, configurando padrão típico de tentativa de pulverização ou divisão de valores para burlar limites operacionais.'}

Teste de resposta malformada: Para validar o tratamento de erros sem depender de uma falha real da API, foi simulada uma resposta inválida contendo um tipo incorreto em red_flags e ausência do campo justificativa. A função validar_parecer() rejeitou a resposta e retornou o erro, impedindo que um parecer inválido fosse utilizado.

In [52]:
resposta_malformada = """
{
    "nivel_risco": "alto",
    "tipologia_suspeita": "Fracionamento",
    "red_flags": "flag de fracionamento"
}
"""

resultado = validar_parecer(resposta_malformada)

print(resultado)

(None, "Schema inválido: 2 validation errors for Parecer\nred_flags\n  Input should be a valid list [type=list_type, input_value='flag de fracionamento', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/list_type\njustificativa\n  Field required [type=missing, input_value={'nivel_risco': 'alto', '...'flag de fracionamento'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing")


Comparação entre os desempenhos e resultados dos dois prompts

In [53]:
cli_a_1["comparacao"]

,prompt,tempo_s,tokens_entrada,tokens_saida,tokens_total,nivel_risco
0,Prompt 1,7.536517,291,128,1593,médio
1,Prompt 2,8.523825,450,174,1866,alto


O Prompt 1, com indicadores mais agregados, classificou o cliente como risco médio e identificou apenas a flag de fracionamento. Já o Prompt 2, com informações adicionais sobre concentração temporal, canais e contrapartes, classificou como alto risco e identificou mais red flags, produzindo uma justificativa mais específica.

Essa maior contextualização teve um custo: o Prompt 2 utilizou 450 tokens de entrada contra 291 e 174 de saída contra 128, além de apresentar maior tempo de resposta (8,52s contra 7,54s). Assim, o Prompt 2 trouxe uma análise mais detalhada e sensível ao comportamento, enquanto o Prompt 1 foi mais simples, rápido e econômico.